In [ ]:
from openai import OpenAI

client = OpenAI(api_key="YOUR_API_KEY")

In [2]:
import os
import json
import pandas as pd
import time
import random
from datetime import datetime, timedelta
import faiss
import numpy as np
import json
import re
from sentence_transformers import SentenceTransformer

c:\Users\Danh\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def _get_store_paths(store_name: str):
    """Tạo đường dẫn file động cho một kho tri thức cụ thể."""
    base_dir = r"D:\finalproject\KLTN\Backend\data\vector_store"
    index_path = os.path.join(base_dir, f"faiss_index_{store_name}.bin")
    docs_path = os.path.join(base_dir, f"documents_{store_name}.json")
    return index_path, docs_path

_stores = {}

def get_store(store_name: str):
    """
    Lấy một kho tri thức cụ thể. Tải từ cache nếu có, nếu không thì xây dựng mới.
    """
    if store_name in _stores:
        return _stores[store_name]

    index_path, docs_path = _get_store_paths(store_name)

    if os.path.exists(index_path) and os.path.exists(docs_path):
        try:
            print(f"Đang tải kho '{store_name}' từ cache...")
            index = faiss.read_index(index_path)
            with open(docs_path, 'r', encoding='utf-8') as f:
                documents = json.load(f)
            print(f"Tải thành công kho '{store_name}' với {index.ntotal} vector.")
            
            store_instance = {"index": index, "documents": documents}
            _stores[store_name] = store_instance
            return store_instance
        except Exception as e:
            print(f"Lỗi khi tải kho '{store_name}' từ cache: {e}. Sẽ xây dựng lại.")

def retrieve(store_name: str, query: str, k: int = 5) -> str:
    """Thực hiện truy vấn trên một kho tri thức chuyên biệt."""
    model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
    store = get_store(store_name)
    if not store or store.get("index") is None:
        print(f"Truy vấn thất bại: Kho tri thức '{store_name}' chưa được khởi tạo.")
        return f"Lỗi: Cơ sở tri thức '{store_name}' không khả dụng."

    index = store["index"]
    documents = store["documents"]
    
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )
    
    try:
        _, indices = index.search(np.array(query_embedding, dtype=np.float32), k)
        retrieved_docs = [documents[i] for i in indices[0]]
        context = "\n---\n".join([doc['content'] for doc in retrieved_docs])
        
        print(f"Đã truy xuất {len(retrieved_docs)} đoạn văn bản từ kho '{store_name}' cho câu hỏi: '{query[:50]}...'")
        return context
    except Exception as e:
        print(f"Lỗi trong quá trình truy xuất từ kho '{store_name}': {e}")
        return "Lỗi: Đã xảy ra sự cố khi tìm kiếm thông tin."

In [4]:
def _define_tools() -> list:
    return [
        {
            "type": "function",
            "function": {
                "name": "answer_general_question",
                "description": "Sử dụng cho MỌI câu hỏi kiến thức chung, định nghĩa, giải thích, thông tin về giai đoạn sinh trưởng, hoặc các câu hỏi không yêu cầu tạo kế hoạch/đề xuất can thiệp cụ thể.",
                "parameters": {"type": "object", "properties": {"question": {"type": "string", "description": "Câu hỏi gốc của người dùng."}}, "required": ["question"]},
            },
        },
        {
            "type": "function",
            "function": {
                "name": "get_nutrient_recommendation",
                "description": "Sử dụng khi người dùng hỏi chung chung về **NGUYÊN TẮC/GỢI Ý** bón phân cho giai đoạn hiện tại (KHÔNG YÊU CẦU TẠO KẾ HOẠCH LƯU DB).",
                "parameters": {"type": "object", "properties": {"problem_description": {"type": "string", "description": "Mô tả của người dùng về vấn đề liên quan đến phân bón."}}, "required": ["problem_description"]},
            },
        },
        {
            "type": "function",
            "function": {
                "name": "get_watering_advice",
                "description": "Sử dụng khi người dùng hỏi chung chung về **NGUYÊN TẮC/GỢI Ý** quản lý nước (KHÔNG YÊU CẦU TẠO KẾ HOẠCH LƯU DB).",
                "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "Câu hỏi của người dùng về việc tưới nước."}}, "required": ["query"]},
            },
        },
        {
            "type": "function",
            "function": {
                "name": "run_proactive_diagnosis",
                "description": "Sử dụng khi người dùng hỏi chung chung về sự có mặt của sâu bệnh (ví dụ: 'lúa có bệnh không?', 'kiểm tra ruộng giúp tôi'). Tool này sẽ tự động phân tích hình ảnh mới nhất từ camera/drone để tìm dấu hiệu bệnh và chỉ nên được gọi khi câu hỏi liên quan đến sức khỏe và sâu bệnh.",
                "parameters": {"type": "object", "properties": {}}, 
            },
        },
        {
            "type": "function",
            "function": {
                "name": "create_new_plan",
                "description": "Sử dụng khi người dùng YÊU CẦU TẠO MỚI một KẾ HOẠCH (bón phân, tưới nước, điều trị). Không sử dụng để trả lời câu hỏi chung. Yêu cầu phải xác định rõ loại kế hoạch.",
                "parameters": {
                    "type": "object", 
                    "properties": {
                        "plan_type": {"type": "string", "enum": ["treatment", "fertilizer", "water"], "description": "Loại kế hoạch cần tạo: treatment, fertilizer, hoặc water."},
                        "disease_name": {"type": "string", "description": "Tên bệnh (dạng tiếng Việt) nếu plan_type là 'treatment', nếu không thì để trống."}
                    }, 
                    "required": ["plan_type"]
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "update_existing_plan",
                "description": "Sử dụng khi người dùng muốn ĐIỀU CHỈNH hoặc PHẢN HỒI về một KẾ HOẠCH (điều trị/bón phân/nước) đã có. Bắt buộc phải xác định được loại kế hoạch muốn cập nhật.",
                "parameters": {
                    "type": "object", 
                    "properties": {
                        "plan_type": {"type": "string", "enum": ["treatment", "fertilizer", "water"], "description": "Loại kế hoạch cần cập nhật."},
                        "user_feedback": {"type": "string", "description": "Phản hồi chi tiết của người dùng (ví dụ: 'Giảm liều lượng xuống 50%')."}
                    }, 
                    "required": ["plan_type", "user_feedback"]
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "get_plan_history",
                "description": "Sử dụng khi người dùng hỏi về các KẾ HOẠCH đã thực thi hoặc đang thực thi trước đó (lịch sử) về một loại can thiệp cụ thể.",
                "parameters": {
                    "type": "object", 
                    "properties": {
                        "plan_type": {"type": "string", "enum": ["treatment", "fertilizer", "water"], "description": "Loại kế hoạch muốn xem lịch sử."},
                        "num_sessions": {"type": "integer", "description": "Số lượng phiên gần nhất muốn xem (mặc định là 3)."}
                    }, 
                    "required": ["plan_type"]
                },
            },
        },
    ]

def _handle_general_qa( farmer_info: dict, question: str, history: list, retrieved_context) -> str:
    print(f"Tool 'answer_general_question' được kích hoạt cho câu hỏi: '{question}'")
    prompt = _build_qa_prompt(farmer_info, question, retrieved_context, history)
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4,
    )
    return response.choices[0].message.content.strip()

# ================== SỬA LẠI CÁC HÀM TOOL ĐỂ CHẠY ĐÚNG ==================

def get_nutrient_recommendation(problem_description: str, farmer_info=None, history=None):
    """Gợi ý nguyên tắc bón phân – dùng kho fertilizer_management"""
    retrieved_context = retrieve("fertilizer_management", problem_description, k=5)
    prompt_question = f"Bác hỏi về phân bón: {problem_description}"
    return _handle_general_qa(farmer_info, prompt_question, history or [], retrieved_context)


def get_watering_advice(query: str, farmer_info=None, history=None):
    """Gợi ý nguyên tắc tưới nước – dùng kho water_management"""
    retrieved_context = retrieve("water_management", query, k=5)
    prompt_question = f"Câu hỏi về tưới nước: {query}"
    return _handle_general_qa(farmer_info, prompt_question, history or [], retrieved_context)


def run_proactive_diagnosis(farmer_info=None, history=None):
    """
    Chẩn đoán chủ động sâu bệnh từ hình ảnh mới nhất
    Dùng kho general_qa hoặc disease_qa tùy bạn đã tạo chưa
    """
    # Giả lập câu hỏi để retrieve (vì người dùng không nhập cụ thể)
    diagnostic_query = "kiểm tra bệnh hại lúa từ hình ảnh ruộng ảnh chụp gần nhất dấu hiệu sâu bệnh"
    retrieved_context = retrieve("general_qa", diagnostic_query, k=5)
    prompt_question = "Kiểm tra giúp tôi xem lúa có bị bệnh hay sâu hại gì không ạ?"
    return _handle_general_qa(farmer_info, prompt_question, history or [], retrieved_context)


def create_new_plan(plan_type: str, disease_name: str = "", farmer_info=None, history=None):
    """Tạo kế hoạch mới – thường không cần retrieve, nhưng vẫn giữ để nhất quán"""
    base_query = {
        "fertilizer": "lập kế hoạch bón phân toàn vụ",
        "water": "lập kế hoạch quản lý nước tưới tiêu",
        "treatment": f"xử lý bệnh {disease_name or 'bệnh hại lúa'}"
    }.get(plan_type, "lập kế hoạch can thiệp nông nghiệp")
    
    retrieved_context = retrieve("general_qa", base_query, k=5)
    prompt_question = f"Tạo kế hoạch mới loại {plan_type}" + (f" cho bệnh {disease_name}" if disease_name else "")
    return _handle_general_qa(farmer_info, prompt_question, history or [], retrieved_context)


def update_existing_plan(plan_type: str, user_feedback: str, farmer_info=None, history=None):
    """Cập nhật kế hoạch hiện có dựa trên phản hồi người dùng"""
    retrieved_context = retrieve("general_qa", f"cập nhật kế hoạch {plan_type} theo phản hồi: {user_feedback}", k=5)
    prompt_question = f"Điều chỉnh kế hoạch {plan_type}: {user_feedback}"
    return _handle_general_qa(farmer_info, prompt_question, history or [], retrieved_context)


def get_plan_history(plan_type: str, num_sessions: int = 3, farmer_info=None, history=None):
    """Xem lại lịch sử các kế hoạch đã thực hiện"""
    retrieved_context = retrieve("general_qa", f"lịch sử kế hoạch {plan_type} gần đây", k=5)
    prompt_question = f"Cho xem lịch sử {num_sessions} lần {plan_type} gần nhất"
    return _handle_general_qa(farmer_info, prompt_question, history or [], retrieved_context)

available_tools = {
    "answer_general_question": _handle_general_qa,
    "get_nutrient_recommendation": get_nutrient_recommendation,
    "get_watering_advice": get_watering_advice,
    "run_proactive_diagnosis": run_proactive_diagnosis,
    "create_new_plan": create_new_plan,
    "update_existing_plan": update_existing_plan,
    "get_plan_history": get_plan_history,
}

def answer_question( farmer_info: dict, question: str, history: list = None):
    if not client:
        return {"error": "Trợ lý AI chưa sẵn sàng.", "history": history or []}
    history = history or []
    if not farmer_info:
        return {"error": "Không tìm thấy thông tin nông hộ.", "history": history}
    
    greetings = ["chào", "hello", "xin chào", "hi"]
    if not history and question.lower().strip() in greetings:
        answer = "Dạ chào bác, tôi là trợ lý nông nghiệp AI. Bác cần tôi giúp gì về việc đồng áng hôm nay ạ?"
        history.append({"role": "user", "content": question})
        history.append({"role": "assistant", "content": answer})
        return {"answer": answer, "history": history}

    messages = [{"role": "system", "content": "Bạn là một trợ lý nông nghiệp AI. Hãy phân tích câu hỏi của người dùng và chọn công cụ phù hợp nhất để trả lời."}]
    messages.extend(history)
    messages.append({"role": "user", "content": question})

    try:
        print(f"QAAgent đang phân tích câu hỏi để chọn tool: '{question}'")
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=_define_tools(),
            tool_choice="auto", 
        )
        response_message = response.choices[0].message
        tool_calls = response_message.tool_calls

        answer = ""
        selected_tool = None
        if tool_calls:
            tool_call = tool_calls[0]
            function_name = tool_call.function.name
            function_to_call = available_tools.get(function_name)
            
            selected_tool = function_name

            if function_to_call:
                function_args = json.loads(tool_call.function.arguments)
                print(f"LLM quyết định gọi tool: '{function_name}' với tham số: {function_args}")

                all_args = {
                    **function_args,
                    'farmer_info': farmer_info,
                    'history': history
                }

                if function_name == "answer_general_question":
                    context = retrieve("general_qa", question, k=5) 
                    all_args["retrieved_context"] = context if context else ""

                try:
                    answer = function_to_call(**all_args)
                except Exception as tool_error:
                    print(f"Lỗi khi thực thi tool {function_name}: {tool_error}")
                    answer = f"Rất tiếc, hiện tại tôi chưa thể thực hiện được yêu cầu này."
            else:
                answer = f"Lỗi: Không tìm thấy hàm thực thi cho tool '{function_name}'."

        history.append({"role": "user", "content": question})
        history.append({"role": "assistant", "content": answer})
        if len(history) > 10:
            history = history[-10:]
        
        return {"answer": answer, "history": history, "selected_tool": selected_tool}

    except Exception as e:
        print(f"Lỗi khi điều phối câu trả lời: {e}", exc_info=True)
        return {"error": "Rất tiếc, đã có lỗi xảy ra. Vui lòng thử lại.", "history": history}

def _build_qa_prompt( farmer_info: dict, question: str, retrieved_context: str, history: list) -> str:
    farmer_json = json.dumps(farmer_info, ensure_ascii=False, indent=2)
    today = datetime.now().date()
    planting_date_str = farmer_info.get("farm_properties", {}).get("planting_date")
    days_since_planting = "không rõ"
    if planting_date_str:
        try:
            planting_date = datetime.strptime(planting_date_str, "%Y-%m-%d").date()
            days_since_planting = (today - planting_date).days
        except ValueError:
            pass 
    history_str = ""
    if history:
        formatted_lines = ["**Lịch sử trò chuyện gần đây:**"]
        for message in history:
            role = "Bác nông dân" if message["role"] == "user" else "Trợ lý AI"
            formatted_lines.append(f"- {role}: {message['content']}")
        history_str = "\n".join(formatted_lines) + "\n"

    prompt = f"""
        **VAI TRÒ & QUY TẮC CỐT LÕI:**
        Bạn là một trợ lý nông nghiệp AI thân thiện, đang nói chuyện trực tiếp với một người nông dân. 
        Nhiệm vụ của bạn là trả lời câu hỏi của họ một cách chính xác, dễ hiểu và CHỈ DỰA TRÊN 
        DỮ LIỆU ĐƯỢC CUNG CẤP TRONG MỤC **'Dữ liệu về nông hộ này'** và **'KIẾN THỨC NỀN'**.

        **CẤM TUYỆT ĐỐI:**
        1. **KHÔNG** suy diễn, thêm, hoặc bịa đặt thông tin không có trong 'Dữ liệu về nông hộ' hoặc 'KIẾN THỨC NỀN'.
        2. Nếu thông tin cần thiết để trả lời không có trong các mục trên, BẮT BUỘC phải trả lời: "Rất tiếc, tôi không có đủ thông tin chi tiết để trả lời chính xác câu hỏi này."
        3. **KHÔNG** trả lời các câu hỏi ngoài lề hoặc không liên quan đến nông nghiệp (Ví dụ: Hỏi về chính trị, thể thao, hay các vấn đề không liên quan đến lúa).

        Dữ liệu về lịch sử trò chuyện
        {history_str}
        **Dữ liệu về nông hộ này (tính đến hôm nay, ngày {today.strftime('%Y-%m-%d')}):**
        ```json
        {farmer_json}
        ```
        **KIẾN THỨC NỀN (Từ cơ sở dữ liệu tri thức):**
        ```
        {retrieved_context}
        ```
        **Thông tin bổ sung:**
        - Hôm nay là ngày thứ {days_since_planting} sau khi gieo sạ.
        
        **Câu hỏi MỚI NHẤT của nông dân:**
        "{question}"
        
        **Yêu cầu đầu ra:**
        Soạn một câu trả lời ngắn gọn, đầy đủ, trực tiếp và thân thiện bằng tiếng Việt. Sử dụng cách xưng hô "bác" và "tôi". Câu trả lời phải mạch lạc, phù hợp với cuộc hội thoại và **chỉ sử dụng thông tin từ các mục đã cho**.

        **Câu trả lời của bạn:**
    """
    return prompt

In [5]:
import time
import json
from collections import Counter
from rouge_score import rouge_scorer
from sklearn.metrics import f1_score
import re

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

GROUND_TRUTH_TOOL = [
    {"id": 1, "question": "Mực nước nên duy trì ở mức nào trong giai đoạn sinh trưởng sinh dưỡng của cây lúa?", "expected_tool": "get_watering_advice"},
    {"id": 2, "question": "Lúa nhà tôi 25 ngày tuổi, giờ có nên bón thúc đợt 1 chưa?", "expected_tool": "get_nutrient_recommendation"},
    {"id": 3, "question": "Kiểm tra giúp tôi xem lúa có bị bệnh gì không bác?", "expected_tool": "run_proactive_diagnosis"},
    {"id": 4, "question": "Tạo cho tôi kế hoạch xử lý bệnh đạo ôn đi", "expected_tool": "create_new_plan"},
    {"id": 5, "question": "Kế hoạch bón phân lần trước nhiều quá, giảm 40% được không?", "expected_tool": "update_existing_plan"},
    {"id": 6, "question": "Cho tôi xem lại lịch sử mấy lần xử lý bệnh gần đây", "expected_tool": "get_plan_history"},
    {"id": 7, "question": "Bệnh đốm nâu do nấm gì gây ra vậy bác?", "expected_tool": "answer_general_question"},
    {"id": 8, "question": "Ruộng lúa của tôi sắp làm đòng rồi, giờ có cần bón đón đòng không bác?", "expected_tool": "get_nutrient_recommendation"},
    {"id": 9, "question": "Bác ơi xem giúp tôi lúa có bị rầy nâu không, chụp hình gửi đây này", "expected_tool": "run_proactive_diagnosis"},
    {"id": 10, "question": "Mùa này trời mưa nhiều, làm sao để phòng bệnh khô vằn hiệu quả?", "expected_tool": "create_new_plan"},
    {"id": 11, "question": "Lúa đang trỗ đều, giữ nước bao nhiêu cm là tốt nhất vậy bác?", "expected_tool": "get_watering_advice"},
    {"id": 12, "question": "Kế hoạch phun thuốc trừ sâu lần trước hiệu quả tốt, đợt này làm giống vậy được không?", "expected_tool": "update_existing_plan"},
    {"id": 13, "question": "Cho tôi xem lại lúa đã bón phân bao nhiêu đợt từ đầu vụ tới giờ với", "expected_tool": "get_plan_history"},
    {"id": 14, "question": "Bệnh bạc lá do vi khuẩn hay nấm vậy bác, trị kiểu gì cho khỏi hẳn?", "expected_tool": "answer_general_question"},
    {"id": 15, "question": "Lúa 45 ngày tuổi mà đẻ nhánh ít quá, có cách nào kích thích đẻ nhánh thêm không?", "expected_tool": "get_nutrient_recommendation"},
    {"id": 16, "question": "Bác kiểm tra giùm ruộng tôi xem có dấu hiệu lem lép hạt không?", "expected_tool": "run_proactive_diagnosis"},
    {"id": 17, "question": "Tạo giúp tôi kế hoạch bón phân toàn vụ cho giống lúa ST25 đi bác", "expected_tool": "create_new_plan"},
    {"id": 18, "question": "Giai đoạn trỗ bông có nên tháo cạn nước vài ngày không hay giữ ngập liên tục?", "expected_tool": "get_watering_advice"},
    {"id": 19, "question": "Lần trước phun thuốc trừ bệnh đạo ôn hơi sớm, lần này dời lại 5 ngày được không?", "expected_tool": "update_existing_plan"},
    {"id": 20, "question": "Cho xem lại lịch sử phun thuốc trừ sâu từ đầu vụ đến giờ bác ơi", "expected_tool": "get_plan_history"},
    {"id": 21, "question": "Ốc bươu vàng đang nhiều quá, dùng thuốc gì diệt mà an toàn cho lúa con?", "expected_tool": "answer_general_question"},
    {"id": 22, "question": "Bác ơi lúa tôi đang giai đoạn nuôi hạt, giờ bón phân gì để hạt chắc mẩy hơn?", "expected_tool": "get_nutrient_recommendation"},
    {"id": 23, "question": "Chụp cho bác xem cái lá lúa nhà tôi vàng vàng này, có bị vàng lá chín sớm không?", "expected_tool": "run_proactive_diagnosis"},
    {"id": 24, "question": "Lúa sắp thu hoạch 10 ngày nữa, làm ơn lập kế hoạch phòng sâu đục thân bông cho chắc ăn", "expected_tool": "create_new_plan"},
    {"id": 25, "question": "Mùa khô năm nay nắng gắt quá, có nên để nước sâu hơn bình thường không bác?", "expected_tool": "get_watering_advice"},
    {"id": 26, "question": "Kế hoạch trừ cỏ lần trước hơi ít thuốc, lần này tăng thêm 20% được không?", "expected_tool": "update_existing_plan"},
    {"id": 27, "question": "Cho tôi xem lại tất cả các lần thay đổi kế hoạch từ đầu vụ tới giờ với bác", "expected_tool": "get_plan_history"},
    {"id": 28, "question": "Sâu cuốn lá nhỏ thì dùng thuốc gì rẻ mà hiệu quả cao vậy bác?", "expected_tool": "answer_general_question"},
    {"id": 29, "question": "Lúa 60 ngày tuổi sắp làm đòng, giờ bón bao nhiêu phân là vừa đủ bác?", "expected_tool": "get_nutrient_recommendation"},
    {"id": 30, "question": "Bác kiểm tra giúp ruộng tôi xem có bị cháy bìa lá do bón phân nhiều quá không?", "expected_tool": "run_proactive_diagnosis"},
    {"id": 31, "question": "Tạo kế hoạch tổng thể từ làm đất tới thu hoạch cho vụ Đông Xuân năm nay đi bác", "expected_tool": "create_new_plan"},
    {"id": 32, "question": "Giai đoạn chín sáp sắp tới có cần tháo nước khô hẳn không hay giữ ẩm nhẹ?", "expected_tool": "get_watering_advice"},
    {"id": 33, "question": "Lần trước bón kali nhiều quá bị hiện tượng đổ ải, lần này giảm còn 70% được không?", "expected_tool": "update_existing_plan"},
    {"id": 34, "question": "Ruộng tôi từng xử lý bệnh mấy lần rồi, cho xem lại chi tiết từng lần được không bác?", "expected_tool": "get_plan_history"},
    {"id": 35, "question": "Rầy phấn trắng đang xuất hiện, có phải do thời tiết ẩm không và trị bằng cách nào?", "expected_tool": "answer_general_question"},
]

REFERENCE_ANSWERS = {
    1: "Trong giai đoạn sinh trưởng sinh dưỡng (7-42 ngày sau sạ), giữ mực nước 5-7 cm, thay nước 2-3 lần, mỗi lần thay giữ cạn 2-3 ngày.",
    2: "Lúa 25 ngày tuổi đang ở cuối giai đoạn đẻ nhánh, bác có thể bón thúc đợt 1 (khoảng 4-5kg urê + 3-4kg kali/sào).",
    3: "Tôi sẽ kiểm tra hình ảnh ruộng mới nhất để xem có dấu hiệu bệnh gì không ạ.", 
    4: "Tôi sẽ tạo kế hoạch xử lý bệnh đạo ôn ngay cho bác.",
    5: "Được ạ, tôi sẽ cập nhật lại kế hoạch bón phân, giảm 40% liều lượng so với lần trước.",
    6: "Đây là lịch sử 3 lần xử lý bệnh gần nhất của ruộng bác...",
    7: "Bệnh đốm nâu trên lúa do nấm Helminthosporium oryzae (hay còn gọi là Cochliobolus miyabeanus) gây ra ạ.",
    8: "Lúa sắp làm đòng (khoảng 45-50 ngày sau sạ) thì nên bón đón đòng ngay, khoảng 3-4kg urê + 4-5kg kali/sào để nuôi đòng to chắc.",
    9: "Đưa hình đây tôi xem kỹ lá và thân lúa có vết chích hút của rầy nâu không nhé.",
    10: "Tôi sẽ lập ngay kế hoạch phòng trừ bệnh khô vằn cho mùa mưa này, tập trung thuốc gốc đồng và quản lý nước hợp lý.",
    11: "Giai đoạn trỗ đều đến chín sáp giữ mực nước 3-5cm, không để ngập sâu quá bông và không để cạn quá 2 ngày liên tục.",
    12: "Được chứ bác, tôi sẽ giữ nguyên hoạt chất và liều lượng như kế hoạch cũ, chỉ điều chỉnh ngày phun theo tình hình thực tế.",
    13: "Từ đầu vụ đến giờ ruộng bác đã bón 3 đợt: đẻ nhánh đợt 1, đợt 2 và đón đòng, tổng cộng khoảng 18kg urê/sào...",
    14: "Bệnh bạc lá do vi khuẩn Pseudomonas glumae gây ra, phải dùng thuốc chứa Kasugamycin hoặc Copper kết hợp cắt nước hợp lý mới khỏi triệt để ạ.",
    15: "Lúa 45 ngày vẫn có thể kích đẻ nhánh thêm bằng cách bón 2-3kg urê/sào + giữ nước nông 3-4cm trong 7-10 ngày tới.",
    16: "Tôi đang xem ảnh vệ tinh và ảnh bác gửi, kiểm tra xem có vết cháy viền lá và hạt lép do lem lép hạt không đây ạ.",
    17: "Tôi tạo ngay kế hoạch bón phân đầy đủ 3 đợt cho giống ST25, đảm bảo năng suất 8-9 tấn/ha.",
    18: "Giai đoạn trỗ bông không tháo cạn hoàn toàn, chỉ cần giữ nước 1-3cm (nước lòng vòng) để tránh thất thoát phấn hoa.",
    19: "Được ạ, tôi sẽ dời lịch phun thuốc phòng đạo ôn cổ bông lại 5 ngày cho phù hợp giai đoạn trỗ của ruộng.",
    20: "Đây là lịch sử phun thuốc trừ sâu từ đầu vụ: đợt 1 (rầy, sâu cuốn lá), đợt 2 (sâu đục thân), đợt 3 vừa rồi (rầy nâu di trú)...",
    21: "Diệt ốc bươu vàng an toàn nhất hiện nay là dùng thuốc Sophia, Sofa, hoặc bả ốc Niclosamide, rải lúc lúa 7-10 ngày tuổi là hiệu quả cao mà ít ảnh hưởng lúa con.",
    22: "Giai đoạn nuôi hạt (sau trỗ 15-25 ngày) nên bón 2-3kg kali/sào + phun phân bón lá Đầu Trâu 007 hoặc Komix để hạt chắc, bóng mẩy hơn ạ.",
    23: "Để tôi xem kỹ hình lá vàng này, có bị vàng lá sinh lý hay vàng lá chín sớm do thiếu kali không nhé.",
    24: "Tôi sẽ lập ngay kế hoạch phun trừ sâu đục thân bông cuối vụ, dùng hoạt chất Chlorantraniliprole hoặc Emamectin kết hợp 2-3 ngày/lần.",
    25: "Mùa khô nắng gắt thì giữ nước 5-8cm vào buổi trưa, sáng sớm và chiều tối có thể giảm xuống 3-4cm để tránh thất thoát nước.",
    26: "Được ạ, tôi sẽ tăng liều thuốc trừ cỏ thêm 20% và điều chỉnh thời điểm phun cho phù hợp tình trạng cỏ hiện tại.",
    27: "Đây là toàn bộ lịch sử thay đổi kế hoạch từ đầu vụ: có 5 lần chỉnh sửa bón phân, 3 lần chỉnh thuốc BVTV và 2 lần quản lý nước...",
    28: "Sâu cuốn lá nhỏ trị rẻ tiền mà hiệu quả thì dùng Virtako, Padan hoặc Regent, phun lúc sâu tuổi 1-2 là diệt sạch ạ.",
    29: "Lúa 60 ngày (đầu làm đòng) bón thúc đón đòng 4kg urê + 5kg kali/sào là chuẩn nhất, chia làm 2 lần cách nhau 5-7 ngày.",
    30: "Tôi đang phân tích ảnh và dữ liệu bón phân gần đây để xem có bị cháy bìa lá do thừa đạm hay thừa phân không đây.",
    31: "Tôi tạo ngay kế hoạch canh tác đầy đủ từ làm đất, giống, phân, thuốc, nước cho vụ Đông Xuân, tối ưu năng suất và chi phí.",
    32: "Giai đoạn chín sáp (sau trỗ 25-30 ngày) tháo cạn nước từ từ, để khô mặt ruộng 7-10 ngày trước thu hoạch cho dễ gặt và hạt khô nhanh.",
    33: "Được chứ bác, tôi giảm ngay liều kali xuống còn 70% so với kế hoạch cũ để tránh hiện tượng đổ ngã và cháy lá.",
    34: "Đây là chi tiết tất cả các lần xử lý bệnh từ đầu vụ: ngày 25/10 đạo ôn, ngày 8/11 khô vằn, ngày 20/11 bạc lá...",
    35: "Đúng rồi ạ, rầy phấn trắng thường bùng phát khi trời ẩm liên tục. Trị tốt nhất bằng Applaud, Chess hoặc Buprofezin phun 2 lần cách nhau 7 ngày."
}

def answer_question_eval(farmer_info: dict, question: str, history: list = None):
    """Phiên bản eval – trả thêm selected_tool để tính accuracy"""
    result = answer_question(farmer_info, question, history or [])
    
    selected_tool = None
    
    if isinstance(result, dict) and "selected_tool" in result:
        selected_tool = result["selected_tool"]
    else:
        answer_text = result.get("answer", "")
        if "LLM quyết định gọi tool:" in answer_text:
            import re
            match = re.search(r"LLM quyết định gọi tool: '([^']+)'", answer_text)
            if match:
                selected_tool = match.group(1)
    
    result["selected_tool"] = selected_tool
    return result


sample_farmer_info = {
    "farmer_id": "eval_001",
    "farm_properties": {
        "planting_date": "2025-10-01",
        "area_ha": 1.2,
        "variety": "OM5451"
    }
}

tool_stats = Counter()
total_questions = len(GROUND_TRUTH_TOOL)
correct_tool = 0
latencies = []

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []
f1_scores = []

print("Bắt đầu đánh giá end-to-end Agent...\n")
for item in GROUND_TRUTH_TOOL:
    qid = item["id"]
    question = item["question"]
    expected_tool = item["expected_tool"]
    ref_answer = REFERENCE_ANSWERS.get(qid, "")
    
    start = time.time()
    response = answer_question_eval(sample_farmer_info, question, [])
    latency = time.time() - start
    latencies.append(latency)
    
    selected_tool = response.get("selected_tool", "unknown")
    answer = response.get("answer", "")
    if isinstance(answer, dict):  # nếu tool trả dict
        answer = str(answer)
    
    # Tool accuracy
    tool_correct = (selected_tool == expected_tool)
    if tool_correct:
        correct_tool += 1
    tool_stats[expected_tool] += 1
    if tool_correct:
        tool_stats[f"{expected_tool}_correct"] += 1
    
    # ROUGE (nếu có reference)
    if ref_answer and answer.strip():
        scores = scorer.score(ref_answer, answer)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)
        
        # Token-level F1 (dễ tính hơn)
        ref_tokens = set(re.findall(r'\w+', ref_answer.lower()))
        pred_tokens = set(re.findall(r'\w+', answer.lower()))
        if ref_tokens or pred_tokens:
            precision = len(ref_tokens & pred_tokens) / len(pred_tokens) if pred_tokens else 0
            recall = len(ref_tokens & pred_tokens) / len(ref_tokens) if ref_tokens else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            f1_scores.append(f1)
    
    status = "TOOL ĐÚNG" if tool_correct else "TOOL SAI"
    print(f"[{qid}] {status} | Tool: {selected_tool.ljust(28)} (mong đợi: {expected_tool}) | {latency:.3f}s")

Bắt đầu đánh giá end-to-end Agent...

QAAgent đang phân tích câu hỏi để chọn tool: 'Mực nước nên duy trì ở mức nào trong giai đoạn sinh trưởng sinh dưỡng của cây lúa?'
LLM quyết định gọi tool: 'get_watering_advice' với tham số: {'query': 'Mực nước nên duy trì ở mức nào trong giai đoạn sinh trưởng sinh dưỡng của cây lúa?'}
Đang tải kho 'water_management' từ cache...
Tải thành công kho 'water_management' với 104 vector.
Đã truy xuất 5 đoạn văn bản từ kho 'water_management' cho câu hỏi: 'Mực nước nên duy trì ở mức nào trong giai đoạn sin...'
Tool 'answer_general_question' được kích hoạt cho câu hỏi: 'Câu hỏi về tưới nước: Mực nước nên duy trì ở mức nào trong giai đoạn sinh trưởng sinh dưỡng của cây lúa?'
[1] TOOL ĐÚNG | Tool: get_watering_advice          (mong đợi: get_watering_advice) | 10.765s
QAAgent đang phân tích câu hỏi để chọn tool: 'Lúa nhà tôi 25 ngày tuổi, giờ có nên bón thúc đợt 1 chưa?'
LLM quyết định gọi tool: 'get_nutrient_recommendation' với tham số: {'problem_description':

In [8]:
avg_latency = sum(latencies) / len(latencies)
p95_latency = sorted(latencies)[int(0.95 * len(latencies)) - 1] if latencies else 0

print("\n" + "="*92)
print("KẾT QUẢ ĐÁNH GIÁ TOÀN DIỆN QAAgent (7 câu test)")
print("="*92)
print(f"Tổng số câu hỏi: {total_questions}")
print(f"→ Tool Selection Accuracy       : {correct_tool / total_questions * 100:5.2f}%  ({correct_tool}/{total_questions})")
print(f"→ ROUGE-1 F1                    : {np.mean(rouge1_scores)*100:5.2f}%" if rouge1_scores else "   N/A")
print(f"→ ROUGE-L F1                    : {np.mean(rougeL_scores)*100:5.2f}%" if rougeL_scores else "   N/A")
print(f"→ Thời gian phản hồi trung bình : {avg_latency:.3f}s")
print(f"→ P95 Latency                   : {p95_latency:.3f}s")
print("\nChi tiết theo công cụ:")
print("-" * 70)

unique_tools = sorted(set(item["expected_tool"] for item in GROUND_TRUTH_TOOL))

print("\nChi tiết theo công cụ:")
print("-" * 70)
for tool_name in unique_tools:
    total = tool_stats[tool_name]
    corr = tool_stats.get(f"{tool_name}_correct", 0)
    acc = corr / total * 100 if total > 0 else 0
    print(f"{tool_name.ljust(30)} | {corr}/{total} câu | {acc:6.2f}%")
print("-" * 70)


KẾT QUẢ ĐÁNH GIÁ TOÀN DIỆN QAAgent (7 câu test)
Tổng số câu hỏi: 35
→ Tool Selection Accuracy       : 88.57%  (31/35)
→ ROUGE-1 F1                    : 41.95%
→ ROUGE-L F1                    : 25.58%
→ Thời gian phản hồi trung bình : 10.048s
→ P95 Latency                   : 13.989s

Chi tiết theo công cụ:
----------------------------------------------------------------------

Chi tiết theo công cụ:
----------------------------------------------------------------------
answer_general_question        | 3/5 câu |  60.00%
create_new_plan                | 4/5 câu |  80.00%
get_nutrient_recommendation    | 5/5 câu | 100.00%
get_plan_history               | 5/5 câu | 100.00%
get_watering_advice            | 5/5 câu | 100.00%
run_proactive_diagnosis        | 5/5 câu | 100.00%
update_existing_plan           | 4/5 câu |  80.00%
----------------------------------------------------------------------
